# 05 · The business case

What the model is worth, in euros, and how far the answer can be trusted.

> **Every cost below is an assumption, not a measurement.** Olist does not publish what a
> negative review costs it, nor the margin on an order, nor how well any intervention works.
> D-04 fixed three numbers — 3 € to intervene, 40 € for a negative review, an intervention that
> works 30% of the time — documented them as assumptions, and committed to measuring how much of
> the conclusion survives changing them. That last part is section 6, and it is the only part
> anybody should act on.

The order of this notebook is not cosmetic. Euros come from multiplying a probability by a cost,
so the probabilities have to be worth multiplying first.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import business, calibrate, config, evaluate, features

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, BAND = "#0b0b0b", "#52514e", "#e8e8e6"
plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 200,
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "grid.color": BAND,
        "grid.linewidth": 0.8,
    }
)

matrix = features.load_features()
y_test = matrix.loc[matrix["split"] == "test", "y"].to_numpy()
scores = business.calibrated_by_moment(matrix, "test")
costs = config.DEFAULT_COSTS

print(f"test block · {len(y_test):,} orders · base rate {y_test.mean():.2%}")
print(f"c_int {costs.c_int} € · C_neg {costs.c_neg} € · e {costs.effectiveness}")
print(f"threshold c_int / (e · C_neg) = {costs.threshold:.2f}")

## 1 · Why calibration comes first

S3 ended by noticing that the constant rule scored a **better Brier** than the t₀ logistic —
0.0935 against 0.0961. A model that ranks nearly twice as well as chance was losing on squared
error, because it learns a 14.70% base rate and scores a 10.19% block. It was calling the test
block 17.2% risky when 10.2% of it went wrong: overstating by two thirds.

Multiply that by 40 € and every euro figure inherits the error. So the probabilities are
corrected first, with a sigmoid fitted on validation — never on training, where the model has
already seen the labels. D-14 has the method comparison; what matters here is the result.

In [ ]:
calibrate.calibration_summary(matrix).round(4)

In [ ]:
TITLES = {"t0": "t0 · at approval", "t1": "t1 · at handover"}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.9), sharex=True, sharey=True)
for ax, moment in zip(axes, ("t0", "t1"), strict=True):
    raw = calibrate.scores_by_block(moment, matrix)["test"]
    calibrated = calibrate.calibrated_scores(moment, matrix)["test"]
    for points, colour, label in ((raw, ORANGE, "raw"), (calibrated, BLUE, "calibrated")):
        table = calibrate.reliability(y_test, points, bins=10)
        gap = calibrate.calibration_error(y_test, points)
        ax.plot(
            table["predicted"],
            table["observed"],
            color=colour,
            marker="o",
            markersize=3.5,
            linewidth=1.6,
            label=f"{label}  (gap {gap:.1%})",
        )
    ax.plot([0, 0.45], [0, 0.45], color=MUTED, linewidth=1.0, linestyle=":")
    ax.text(0.30, 0.325, "perfect", color=MUTED, fontsize=7.5, rotation=38)
    ax.set_title(TITLES[moment], loc="left", color=INK)
    ax.set_xlabel("predicted risk")
    ax.grid(alpha=0.5)
    ax.set_xlim(0, 0.45)
    ax.set_ylim(0, 0.45)
    ax.legend(frameon=False, fontsize=7.5, loc="upper left")

axes[0].set_ylabel("orders that really ended badly")
fig.suptitle(
    "Calibration on the test block · every point is a tenth of the orders, "
    "below the line means the model overstates risk",
    x=0.005,
    ha="left",
    color=MUTED,
    fontsize=8.5,
)
fig.tight_layout()
fig.savefig(config.FIGURES / "calibration.png", bbox_inches="tight")

Both calibrated models now beat the constant rule on Brier, which neither did before. A residual
bias remains — 11.5% predicted against 10.2% observed at t₀ — and it is left alone on purpose:
the calibrator learns an 11.86% block and is applied to a 10.19% one, because the drift does not
stop. Recalibrating on test would remove it and would be exactly the cheat this project has
avoided everywhere else. It is a monitoring problem, and it belongs to phase 4.

## 2 · Without a model, intervening destroys value

Before asking what the model is worth, it is worth asking whether anything is needed at all.
Acting costs 3 € per order and saves 0.30 × 40 = 12 € on the ones that would have gone wrong —
10.19% of them. That is 1.22 € of expected benefit against 3 € of cost.

In [ ]:
business.blanket_policies(y_test, costs).round(2)

**Blanket intervention is not expensive, it is ruinous: 33,180 € destroyed on this block alone.**
That number, rather than any metric, is what the model exists to fix.

## 3 · The operating point

Act where the calibrated risk clears `c_int / (e · C_neg)` = 0.25 — a threshold derived from the
cost ratio alone, fixed before any outcome was seen.

In [ ]:
business.policy_table(matrix).round(4)

In [ ]:
per_1000 = 1000 / len(y_test)
STYLE = {"t0": (ORANGE, "t0 · at approval"), "t1": (BLUE, "t1 · at handover")}

fig, (left, right) = plt.subplots(1, 2, figsize=(9.2, 3.8))

cuts = np.linspace(0.05, 0.6, 60)
for moment, (colour, label) in STYLE.items():
    curve = [evaluate.net_savings(y_test, scores[moment], c, costs) * per_1000 for c in cuts]
    left.plot(cuts, curve, color=colour, linewidth=1.8, label=label)
    here = evaluate.net_savings(y_test, scores[moment], costs.threshold, costs) * per_1000
    left.plot([costs.threshold], [here], marker="o", color=colour, markersize=5)
    left.annotate(
        f"{here:.0f} €",
        (costs.threshold, here),
        textcoords="offset points",
        xytext=(8, -2),
        color=colour,
        fontsize=8,
        fontweight="bold",
    )

left.axvline(costs.threshold, color=MUTED, linewidth=0.9, linestyle=":")
left.text(
    0.258, -8, "0.25 = c_int / (e · C_neg)\nfixed before any outcome", color=MUTED, fontsize=7
)
left.axhline(0, color=MUTED, linewidth=0.8)
left.set_xlabel("threshold on calibrated risk")
left.set_ylabel("€ saved per 1,000 orders")
left.grid(alpha=0.5)
left.set_xlim(0.05, 0.6)
left.set_ylim(-20, 80)
left.set_title("The operating point", loc="left", color=INK)
left.legend(frameon=False, fontsize=7.5, loc="upper right")

truth = np.linspace(0.10, 0.45, 60)
for moment, (colour, label) in STYLE.items():
    acted = scores[moment] >= costs.threshold
    caught, flagged = int((acted & y_test).sum()), int(acted.sum())
    right.plot(
        truth * 100,
        (caught * truth * costs.c_neg - flagged * costs.c_int) * per_1000,
        color=colour,
        linewidth=1.8,
        label=label,
    )
    crossing = business.break_even(y_test, scores[moment])["effectiveness"]
    right.plot([crossing * 100], [0], marker="o", color=colour, markersize=5)
    right.annotate(
        f"{crossing:.1%}",
        (crossing * 100, 0),
        textcoords="offset points",
        xytext=(-40, 8) if moment == "t1" else (6, -20),
        color=colour,
        fontsize=8,
        fontweight="bold",
    )

right.axhline(0, color=MUTED, linewidth=0.8)
right.axvline(30, color=MUTED, linewidth=0.9, linestyle=":")
right.text(30.6, 88, "what D-04\nassumed", color=MUTED, fontsize=7)
right.fill_betweenx([-40, 110], 10, 23.04, color=BAND, alpha=0.55, zorder=0)
right.text(11, 72, "both moments\ndestroy value", color=MUTED, fontsize=7.5)
right.fill_betweenx([-40, 110], 23.04, 25.39, color=BAND, alpha=0.28, zorder=0)
right.text(23.4, 40, "only t1\nstill pays", color=MUTED, fontsize=7, rotation=90)
right.set_xlabel("how well the intervention really works (%)")
right.set_ylabel("€ saved per 1,000 orders")
right.grid(alpha=0.5)
right.set_xlim(10, 45)
right.set_ylim(-40, 110)
right.set_title("Keep the threshold, change reality", loc="left", color=INK)
right.legend(frameon=False, fontsize=7.5, loc="lower right")

fig.suptitle(
    "Test block · 18,664 orders · costs are the assumptions of D-04, not measurements",
    x=0.005,
    ha="left",
    color=MUTED,
    fontsize=8.5,
)
fig.tight_layout()
fig.savefig(config.FIGURES / "business_case.png", bbox_inches="tight")

**t₁ saves 2.5× what t₀ saves** — 58.51 € against 23.31 € per thousand orders. At this volume,
roughly 74,700 orders a year, that is about 4,400 € and 1,700 € annually.

Which is little money, and saying so is part of the work. Two things follow from it rather than
from wishing it were larger: the figures scale linearly with volume and with `C_neg`, and the
policy is deliberately narrow — with these costs only 4–6% of orders clear the bar, so a low
recall is the design working, not failing. Nobody is trying to catch every negative review, only
the ones where acting pays.

## 4 · The fixed threshold against hindsight

In [ ]:
business.threshold_comparison(matrix).round(3)

The empirical optima land between 0.23 and 0.33, around the 0.25 that theory fixed in advance.
**That is a calibration check, not a missed opportunity:** a perfectly calibrated model puts its
optimum exactly at `c_int / (e · C_neg)`, so the distance reads as the residual drift section 1
left in place. The threshold that ships stays the fixed one — it was chosen without looking at
any outcome, and that is the only reason it can be trusted on next month's orders.

## 5 · The question D-04 deferred

The cost matrix is identical at both moments **on purpose**: fix the economics, vary only the
information, and the difference measures the value of the signal. That is the right way to
measure, and the wrong way to decide.

Because the moments are not equally actionable. At t₀ nothing has moved and there are real levers
— reroute, change carrier, warn the seller. At t₁ the parcel is already travelling and what is
left is communication and goodwill. So: hold t₁ at 0.30, let t₀ be more effective, and let the
threshold follow, since `c_int / (e · C_neg)` says it must.

In [ ]:
business.effectiveness_scenarios(matrix).round(2)

The two cross at about **0.44**. In one defensible sentence:

> t₁'s signal advantage — +0.048 PR-AUC — is cancelled if intervening at approval is some 14
> points more effective than intervening after dispatch.

Whether that is plausible is an operations question, not a data one, and this dataset cannot
answer it. Putting the number on the table is what it can do.

*(The curve is not monotone — 0.42 gives 1,052 € and 0.43 gives 1,036 €. Moving the effectiveness
moves the threshold, which changes the set of flagged orders in steps. It is a step function, not
estimation noise.)*

## 6 · How far does this hold?

Two different questions hide inside "what if the costs are wrong", and only one has an
interesting answer.

**Re-derive the threshold whenever an assumption changes, and the policy cannot lose money.** It
only acts where `c_int < e · p · C_neg`, so a pessimistic assumption does not destroy value — it
raises the bar until nothing clears it. Which is why the savings below travel with the share of
orders flagged: a small euro figure means two very different things, and euros alone cannot tell
a programme that works badly from one that never runs.

In [ ]:
grid = business.sensitivity(matrix, moment="t1", c_int_values=(3.0,))
savings = grid.pivot_table(index="c_neg", columns="effectiveness", values="savings_per_1000_orders")
flagged = grid.pivot_table(index="c_neg", columns="effectiveness", values="flagged_share") * 100

print("€ saved per 1,000 orders, t1")
print(savings.round(1).to_string())
print("\n% of orders acted on")
print(flagged.round(1).to_string())

At `C_neg` = 15 € and 15% effectiveness the model flags **no orders at all**. The programme does
not fail, it disappears.

The realistic failure is the other one: the threshold was set believing 0.30, and reality is
lower, so the same orders keep being flagged and stop paying for themselves. With the acting set
fixed, savings are linear in effectiveness and cross zero at

$$e^* = \frac{c_{int}}{C_{neg} \times \text{precision}}$$

a frontier that depends on the cost ratio and on the precision the model reaches — and on nothing
about the size of the block or its base rate.

In [ ]:
pd.DataFrame(
    [{"moment": moment, **business.break_even(y_test, scores[moment])} for moment in ("t0", "t1")]
).round(4)

## What this project concludes

**The case holds, and it holds narrowly.** 0.30 was assumed; t₀ breaks even at 0.254 and t₁ at
0.230. The assumption can be 15% too optimistic at t₀ before the programme starts destroying
value.

That thinness is structural rather than a defect. The threshold sits exactly at break-even, so
the marginal flagged order contributes nothing by construction and all the profit comes from
orders well clear of the bar. Any error in the assumptions eats the margin quickly.

Which leads somewhere that is not about the model at all:

> **Before building this for real, measure the effectiveness of the intervention.** An A/B test
> over a few thousand flagged orders answers the one question everything rests on. No improvement
> in PR-AUC substitutes for it.

And it gives the honest reason to keep working on the model, which is not "a better score":
raising precision at the operating point from 32.6% to 40% would move the break-even from 23.0%
to **18.8%**. That is buying margin against an assumption nobody has measured yet — a claim an
operations director can act on, in a way that a PR-AUC cannot be.